# Exploración de la API de ESIOS

Objetivo:

- Comprender la estructura del JSON.
- Identificar los campos relevantes.
- Detectar problemas de calidad.
- Definir las transformaciones necesarias para el pipeline.

In [1]:
import requests
import json
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
API_KEY = os.getenv("ESIOS_API_KEY")

In [ ]:
#LLamada simple a la API de ESIOS para obtener una lista de indicadores
HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY
}
url = "https://api.esios.ree.es/indicators"
response = requests.get(
    url,
    headers=HEADERS,
)
response_json = response.json()
pprint(response_json)

{'indicators': [{'description': '<p>Es el programa de energía diario, con '
                                'desglose horario, de las diferentes Unidades '
                                'de Programación correspondientes a ventas y '
                                'adquisiciones de energía en el sistema '
                                'eléctrico peninsular español. En concreto '
                                'este indicador se refiere a las unidades de '
                                'programación con tipo de producción '
                                'hidráulica UGH.</p><p>Este programa es '
                                'establecido por el OS a partir de la casación '
                                'del OM y de las nominaciones de programas de '
                                'todas y cada una de las Unidades de '
                                'Programación que le han sido comunicadas por '
                                'los sujetos titulares de dichas Unidades de '

In [7]:
# LLamada a la API de ESIOS para obtener los datos de un indicador específico
indicator_id = 1293
url = f"https://api.esios.ree.es/indicators/{indicator_id}"
PARAMS = {
    "start_date": "2026-01-01T00:00:00",
    "end_date": "2026-01-31T23:59:59",
    "geo_ids": "ES",
    "time_trunc": "hour"
}
response = requests.get(
    url,
    headers=HEADERS,
    params=PARAMS
)
response_json = response.json()
pprint(response_json)

{'indicator': {'composited': False,
               'disaggregated': False,
               'geos': [{'geo_id': 8741, 'geo_name': 'Península'}],
               'id': 1293,
               'magnitud': [{'id': 20, 'name': 'Potencia'}],
               'name': 'Demanda real',
               'short_name': 'Demanda real',
               'step_type': 'linear',
               'tiempo': [{'id': 219, 'name': 'Cinco minutos'}],
               'values': [{'datetime': '2026-01-01T00:00:00.000+01:00',
                           'datetime_utc': '2025-12-31T23:00:00Z',
                           'geo_id': 8741,
                           'geo_name': 'Península',
                           'tz_time': '2025-12-31T23:00:00.000Z',
                           'value': 285837.0},
                          {'datetime': '2026-01-01T01:00:00.000+01:00',
                           'datetime_utc': '2026-01-01T00:00:00Z',
                           'geo_id': 8741,
                           'geo_name': 'Península',
 

In [9]:
# Guardar la respuesta en un archivo JSON data/raw/sample_response.json
with open("../data/raw/sample_response.json", "w") as f:
    json.dump(response_json, f, indent=4)


In [10]:
# Convertir los datos de serie temporal a un DataFrame de pandas
df = pd.DataFrame(response_json['indicator']['values']) 
print(df.head()) # Mostrar las primeras filas del DataFrame
print(df.info()) # Mostrar información del DataFrame
print(df.describe()) # Mostrar estadísticas descriptivas del DataFrame 

      value                       datetime          datetime_utc  \
0  285837.0  2026-01-01T00:00:00.000+01:00  2025-12-31T23:00:00Z   
1  275880.0  2026-01-01T01:00:00.000+01:00  2026-01-01T00:00:00Z   
2  259921.0  2026-01-01T02:00:00.000+01:00  2026-01-01T01:00:00Z   
3  245445.0  2026-01-01T03:00:00.000+01:00  2026-01-01T02:00:00Z   
4  234926.0  2026-01-01T04:00:00.000+01:00  2026-01-01T03:00:00Z   

                    tz_time  geo_id   geo_name  
0  2025-12-31T23:00:00.000Z    8741  Península  
1  2026-01-01T00:00:00.000Z    8741  Península  
2  2026-01-01T01:00:00.000Z    8741  Península  
3  2026-01-01T02:00:00.000Z    8741  Península  
4  2026-01-01T03:00:00.000Z    8741  Península  
<class 'pandas.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   value         744 non-null    float64
 1   datetime      744 non-null    str    
 2   datetime_utc  744 non-null 

In [11]:
df.describe()

,value,geo_id
count,744.000000,744.0
mean,368605.494624,8741.0
std,68460.893895,0.0
min,225474.000000,8741.0
25%,307642.000000,8741.0
50%,373880.500000,8741.0
75%,433596.250000,8741.0
max,493045.000000,8741.0


| Columna  | Tipo     | ¿Se conservará? | Motivo                 |
| -------- | -------- | --------------- | ---------------------- |
| datetime | str | No             | Evitar problemas con cambios de horario        |
| value    | float   | Sí              | Métrica                |
| geo_id   | int      | Sí              | Dimensión geográfica   |
| geo_name     | str     | Si            | Se conserva pero sera normalizado posteriormente |
| datetime_utc    | str     | Si            | Clave temporal universal |
| tz_time    | str      | No           | innecesario ya que se utilizara UTC |


## Analisis de calidad de los datos

In [13]:
df.isnull().sum() # Contar valores nulos en cada columna

value           0
datetime        0
datetime_utc    0
tz_time         0
geo_id          0
geo_name        0
dtype: int64

In [14]:
df.duplicated().sum() # Contar valores duplicados en el DataFrame

np.int64(0)

In [15]:
df.dtypes # Mostrar los tipos de datos de cada columna

value           float64
datetime            str
datetime_utc        str
tz_time             str
geo_id            int64
geo_name            str
dtype: object

## Conclusiones

- La informacion util del indicador se encuentra en la coleccion values
- Sera necesario eliminar las columnas innecesarias
- sera necesario convertir fechas a formatos estandar
- No se detectaron valores nulos ni registros duplicados. No obstante, el pipeline incorporará validaciones de calidad para garantizar la integridad de los datos ante posibles cambios en la API o incidencias futuras